# RLHF notebook — intended for **Kaggle**

Kaggle ships `trl`, `transformers`, `accelerate`, and `datasets`; no install cell or runtime restart is needed. Enable **GPU T4 x2** under Settings → Accelerator, then **Run All** (see README).

In [ ]:
import os, time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, pipeline, set_seed
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead, create_reference_model

device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(42)
print("device:", device)

In [ ]:
base_model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLMWithValueHead.from_pretrained(base_model_name).to(device)
ref_model = create_reference_model(model)

print("Loaded:", base_model_name)

In [ ]:
reward_model_name = "lvwerra/distilbert-imdb"
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model=reward_model_name,
    device=0 if device == "cuda" else -1,
    truncation=True,
    max_length=512,
    return_all_scores=True,
    function_to_apply="none",
)

def compute_rewards(text_list, scale=1.0):
    outs = sentiment_pipe(text_list, batch_size=16)
    rewards = []
    for o in outs:
        score_map = {d["label"].upper(): float(d["score"]) for d in o}
        pos = score_map.get("POSITIVE", score_map.get("LABEL_1", 0.0))
        neg = score_map.get("NEGATIVE", score_map.get("LABEL_0", 0.0))
        r = (pos - neg) * scale
        rewards.append(r)
    return rewards

print(compute_rewards(["I loved it!", "I hated it!"]))

In [ ]:
raw = load_dataset("imdb", split="train").shuffle(seed=42).select(range(2000))

def build_prompt(ex, max_chars=200):
    snippet = ex["text"].replace("\n", " ").strip()[:max_chars]
    prompt = (
        "Write a movie review in a clearly POSITIVE tone about the following content.\n"
        f"Content: {snippet}\n"
        "Positive Review:"
    )
    return {"prompt": prompt}

ds = raw.map(build_prompt, remove_columns=raw.column_names)

max_prompt_len = 256
def tokenize_fn(ex):
    out = tokenizer(ex["prompt"], truncation=True, max_length=max_prompt_len, padding=False)
    return out

ds = ds.map(tokenize_fn, remove_columns=["prompt"])
ds.set_format(type="torch")
print(ds[0].keys(), len(ds))

In [ ]:
ppo_config = PPOConfig(
    learning_rate=1e-6,
    batch_size=8,
    mini_batch_size=4,
    ppo_epochs=4,
    gradient_accumulation_steps=1,
    target_kl=0.2,
    # seed= removed: not supported in trl==0.11.4
)

def collator(data):
    return tokenizer.pad(data, padding=True, return_tensors="pt")

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=ds,
    data_collator=collator,
)

print("PPOTrainer ready. Steps:", len(ppo_trainer.dataloader))

In [ ]:
def generate_samples(prompts, max_new_tokens=80):
    model.eval()
    outs = []
    for p in prompts:
        inputs = tokenizer(p, return_tensors="pt", truncation=True, max_length=max_prompt_len).to(device)
        with torch.no_grad():
            gen = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                top_p=0.95,
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        prompt_len = inputs["input_ids"].shape[-1]
        completion_ids = gen[0, prompt_len:]
        outs.append(tokenizer.decode(completion_ids, skip_special_tokens=True).strip())
    return outs

sample_prompts = [
    "Write a movie review in a clearly POSITIVE tone about the following content.\nContent: A story about a family overcoming hardships.\nPositive Review:",
    "Write a movie review in a clearly POSITIVE tone about the following content.\nContent: An action movie with a hero saving the city.\nPositive Review:",
]

before = generate_samples(sample_prompts)
before_r = compute_rewards(before)
for i,(t,r) in enumerate(zip(before,before_r),1):
    print("="*80)
    print(f"[BEFORE {i}] reward={r:+.4f}\n{t[:900]}")
print("mean before:", sum(before_r)/len(before_r))

In [ ]:
import time
import numpy as np
from trl.core import LengthSampler

model = model.to(device)

output_min_length = 8
output_max_length = 24
output_length_sampler = LengthSampler(output_min_length, output_max_length)

generation_kwargs = dict(
    min_length=-1,
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id,
)

total_steps = 200
t0 = time.time()
model.train()

for step, batch in enumerate(ppo_trainer.dataloader):
    if step >= total_steps:
        break

    input_ids = batch["input_ids"].to(device)
    attn = batch["attention_mask"].to(device)
    query_tensors = [ids[mask.bool()] for ids, mask in zip(input_ids, attn)]

    gen_len = output_length_sampler()
    generation_kwargs["max_new_tokens"] = gen_len

    t_gen0 = time.time()
    outs = ppo_trainer.generate(query_tensors, return_prompt=False, **generation_kwargs)
    t_gen1 = time.time()

    if isinstance(outs, torch.Tensor):
        outs = [x for x in outs]

    response_tensors = []
    for out in outs:
        out = out.squeeze()
        response_tensors.append(out)

    gen_texts = [tokenizer.decode(r, skip_special_tokens=True) for r in response_tensors]

    t_rw0 = time.time()
    rewards = compute_rewards(gen_texts)
    t_rw1 = time.time()

    rewards_tensors = [torch.tensor(float(r), device=device) for r in rewards]

    stats = ppo_trainer.step(query_tensors, response_tensors, rewards_tensors)

    mean_reward = float(np.mean(rewards))
    if step % 5 == 0:
        # Fixed: read KL from stats dict instead of undefined get_kl()
        kl = stats.get("objective/kl", stats.get("ppo/mean_non_score_reward", None))
        if kl is None:
            print(f"step={step:04d} mean_reward={mean_reward:+.4f} gen={t_gen1-t_gen0:.2f}s reward={t_rw1-t_rw0:.2f}s")
        else:
            print(f"step={step:04d} mean_reward={mean_reward:+.4f} kl={float(kl):+.4f} gen={t_gen1-t_gen0:.2f}s reward={t_rw1-t_rw0:.2f}s")

print("done, time:", round(time.time() - t0, 1), "s")

In [ ]:
after = generate_samples(sample_prompts)
after_r = compute_rewards(after)
for i,(t,r) in enumerate(zip(after,after_r),1):
    print("="*80)
    print(f"[AFTER {i}] reward={r:+.4f}\n{t[:900]}")

print("mean before:", sum(before_r)/len(before_r))
print("mean after :", sum(after_r)/len(after_r))

In [ ]:
save_dir = "rlhf_positive_gpt2_trl"
os.makedirs(save_dir, exist_ok=True)
ppo_trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved:", save_dir)

In [ ]:
import torch
import numpy as np
from datasets import load_dataset

model = model.to(device)
ref_model = ref_model.to(device)

def build_prompt_eval(text, max_chars=200):
    snippet = text.replace("\n", " ").strip()[:max_chars]
    return (
        "Write a movie review in a clearly POSITIVE tone about the following content.\n"
        f"Content: {snippet}\n"
        "Positive Review:"
    )

EVAL_N = 200
test_ds = load_dataset("imdb", split="test").shuffle(seed=123).select(range(EVAL_N))
eval_prompts = [build_prompt_eval(t) for t in test_ds["text"]]

@torch.no_grad()
def generate_from(model_obj, prompts, max_prompt_len=256, max_new_tokens=80, batch_size=8):
    model_obj.eval()
    outputs = []
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        enc = tokenizer(
            batch_prompts,
            return_tensors="pt",
            truncation=True,
            max_length=max_prompt_len,
            padding=True,
        ).to(device)
        gen = model_obj.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.95,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
        prompt_len = enc["input_ids"].shape[-1]
        completion_ids = gen[:, prompt_len:]
        outputs.extend(t.strip() for t in tokenizer.batch_decode(completion_ids, skip_special_tokens=True))
    return outputs

baseline_texts = generate_from(ref_model, eval_prompts, max_new_tokens=80, batch_size=8)
trained_texts  = generate_from(model, eval_prompts, max_new_tokens=80, batch_size=8)

baseline_rewards = np.array(compute_rewards(baseline_texts), dtype=np.float32)
trained_rewards  = np.array(compute_rewards(trained_texts), dtype=np.float32)
delta = trained_rewards - baseline_rewards

print("Eval size:", len(eval_prompts))
print("Baseline mean reward:", float(baseline_rewards.mean()))
print("Trained   mean reward:", float(trained_rewards.mean()))
print("Mean delta (trained - baseline):", float(delta.mean()))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(7,4))
plt.hist(baseline_rewards, bins=30, alpha=0.6, label="Baseline (ref_model)")
plt.hist(trained_rewards,  bins=30, alpha=0.6, label="Trained (RLHF PPO)")
plt.xlabel("Reward (sentiment score)")
plt.ylabel("Count")
plt.title("Reward Distribution: Baseline vs Trained")
plt.legend()
plt.show()

means = [baseline_rewards.mean(), trained_rewards.mean()]
stds  = [baseline_rewards.std(),  trained_rewards.std()]

plt.figure(figsize=(5,4))
plt.bar(["Baseline", "Trained"], means, yerr=stds)
plt.ylabel("Reward")
plt.title("Mean Reward (±1 std)")
plt.show()

plt.figure(figsize=(5,5))
plt.scatter(baseline_rewards, trained_rewards, alpha=0.6)
mn = float(min(baseline_rewards.min(), trained_rewards.min()))
mx = float(max(baseline_rewards.max(), trained_rewards.max()))
plt.plot([mn, mx], [mn, mx])
plt.xlabel("Baseline reward")
plt.ylabel("Trained reward")
plt.title("Per-prompt Reward: Baseline vs Trained")
plt.show()

In [ ]:
import numpy as np
import textwrap

top_k = 5
top_idx = np.argsort(-delta)[:top_k]

for rank, idx in enumerate(top_idx, 1):
    prompt_snip = eval_prompts[idx].split("Content:",1)[-1].split("\nPositive Review:",1)[0].strip()

    print("="*100)
    print(f"[TOP IMPROVEMENT #{rank}] idx={idx}")
    print(f"Baseline reward: {baseline_rewards[idx]:+.4f}")
    print(f"Trained  reward: {trained_rewards[idx]:+.4f}")
    print(f"Delta          : {delta[idx]:+.4f}")
    print("-"*100)
    print("PROMPT (content snippet):")
    print(textwrap.fill(prompt_snip[:300], width=100))
    print("-"*100)
    print("BASELINE OUTPUT:")
    print(baseline_texts[idx][:1000])
    print("-"*100)
    print("TRAINED OUTPUT:")
    print(trained_texts[idx][:1000])

In [ ]:
# ============================================================
# BERT 过程：使用 DistilBERT 情感分类器 (lvwerra/distilbert-imdb)
# 对 baseline / trained 输出做显式标签 + 置信度对比，
# 生成 ≥15 条 before/after 对比分析表。
# 依赖前面 Cell 中定义的:
#   sentiment_pipe, eval_prompts, baseline_texts, trained_texts,
#   baseline_rewards, trained_rewards, delta
# ============================================================
import pandas as pd
import textwrap

def bert_classify(text_list):
    """用 DistilBERT 对每条文本输出 (label, confidence, pos-neg margin)。"""
    outs = sentiment_pipe(text_list, batch_size=16)
    results = []
    for o in outs:
        score_map = {d["label"].upper(): float(d["score"]) for d in o}
        pos = score_map.get("POSITIVE", score_map.get("LABEL_1", 0.0))
        neg = score_map.get("NEGATIVE", score_map.get("LABEL_0", 0.0))
        label = "POSITIVE" if pos >= neg else "NEGATIVE"
        confidence = max(pos, neg)
        margin = pos - neg
        results.append((label, confidence, margin))
    return results

N_TABLE = min(20, len(eval_prompts))  # 至少 15 条，这里取 20 条
assert N_TABLE >= 15, f"需要至少 15 条，但 eval_prompts 只有 {len(eval_prompts)} 条"

base_cls = bert_classify(baseline_texts[:N_TABLE])
trn_cls  = bert_classify(trained_texts[:N_TABLE])

rows = []
for i in range(N_TABLE):
    snippet = eval_prompts[i].split("Content:", 1)[-1].split("\nPositive Review:", 1)[0].strip()
    if base_cls[i][0] == "NEGATIVE" and trn_cls[i][0] == "POSITIVE":
        flip = "NEG→POS"
    elif base_cls[i][0] == "POSITIVE" and trn_cls[i][0] == "NEGATIVE":
        flip = "POS→NEG"
    else:
        flip = ""
    rows.append({
        "idx": i,
        "prompt": (snippet[:55] + "...") if len(snippet) > 55 else snippet,
        "before_label": base_cls[i][0],
        "before_conf":  round(base_cls[i][1], 3),
        "before_rwd":   round(float(baseline_rewards[i]), 3),
        "after_label":  trn_cls[i][0],
        "after_conf":   round(trn_cls[i][1], 3),
        "after_rwd":    round(float(trained_rewards[i]), 3),
        "delta":        round(float(delta[i]), 3),
        "flip":         flip,
    })

df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 220)
print(f"BERT (DistilBERT-IMDB) before/after 对比分析表（共 {len(df)} 条）")
print("=" * 120)
print(df.to_string(index=False))
print("=" * 120)

flips_n2p = int(((df["before_label"] == "NEGATIVE") & (df["after_label"] == "POSITIVE")).sum())
flips_p2n = int(((df["before_label"] == "POSITIVE") & (df["after_label"] == "NEGATIVE")).sum())
both_pos  = int(((df["before_label"] == "POSITIVE") & (df["after_label"] == "POSITIVE")).sum())
both_neg  = int(((df["before_label"] == "NEGATIVE") & (df["after_label"] == "NEGATIVE")).sum())
improved  = int((df["delta"] > 0).sum())

print("摘要统计:")
print(f"  NEG → POS 翻转 : {flips_n2p:>3d} / {len(df)}")
print(f"  POS → NEG 翻转 : {flips_p2n:>3d} / {len(df)}")
print(f"  始终 POSITIVE  : {both_pos:>3d} / {len(df)}")
print(f"  始终 NEGATIVE  : {both_neg:>3d} / {len(df)}")
print(f"  reward 上升条数: {improved:>3d} / {len(df)}")
print(f"  平均 before reward = {df['before_rwd'].mean():+.4f}")
print(f"  平均 after  reward = {df['after_rwd'].mean():+.4f}")
print(f"  平均 delta         = {df['delta'].mean():+.4f}")

try:
    from IPython.display import display
    display(df)
except Exception:
    pass

print("\n详细 before/after 文本对比（前 5 条）")
print("=" * 120)
for i in range(min(5, N_TABLE)):
    print(f"\n[#{i}]  before={base_cls[i][0]}({base_cls[i][1]:.3f})  "
          f"after={trn_cls[i][0]}({trn_cls[i][1]:.3f})  delta={float(delta[i]):+.4f}")
    print("BEFORE:", textwrap.shorten(baseline_texts[i].replace("\n", " "), width=320, placeholder=" ..."))
    print("AFTER :", textwrap.shorten(trained_texts[i].replace("\n", " "), width=320, placeholder=" ..."))

In [ ]:
model.save_pretrained("ppo_positive_imdb")
tokenizer.save_pretrained("ppo_positive_imdb")
print("Final model saved to ppo_positive_imdb/")